# Scouter - Match Analytics Pipeline (Google Colab)
**Match:** Finale Scudetto 2023-24 Virtus Bologna - Olimpia Milano G1 (FULL GAME)
**Video URL:** https://www.youtube.com/watch?v=37Lw3KKzlns

This notebook runs the Scouter multi-agent pipeline on Google Colab using GPU acceleration.

In [ ]:
# 1. Verify GPU acceleration
!nvidia-smi

In [ ]:
# 2. Install required dependencies
!pip install -q -U yt-dlp yt-dlp-youtube-oauth2 opencv-python ffmpeg-python ultralytics easyocr librosa pydub matplotlib

In [ ]:
# 3. Set match URL and working directories
import os
from pathlib import Path

MATCH_URL = "https://www.youtube.com/watch?v=37Lw3KKzlns"
DATA_DIR = Path("data/match_g1_milano_virtus")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Target match:", MATCH_URL)
print("Output directory:", DATA_DIR)

In [ ]:
# 4. Agent 1 (Cameraman): Download full video and extract mono audio track
# Note: In Colab, datacenter bandwidth downloads Full HD / 720p streams in ~30 seconds.
# use_oauth2=True bypasses Google Cloud datacenter bot protection via a one-time device code.
from src.agents.cameraman import CameramanAgent

cameraman = CameramanAgent(output_dir=DATA_DIR)
metadata = cameraman.process(MATCH_URL, resolution="720", use_oauth2=True)
print("Video and audio ingestion complete:", metadata)

In [ ]:
# 5. Scoreboard ROI Calibration & Frame Inspection
import cv2
import matplotlib.pyplot as plt

video_path = DATA_DIR / "raw_video.mp4"
cap = cv2.VideoCapture(str(video_path))

# Read frame at minute 5 (300 sec * fps)
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
cap.set(cv2.CAP_PROP_POS_FRAMES, int(300 * fps))
ret, frame = cap.read()
cap.release()

if ret:
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(12, 7))
    plt.imshow(frame_rgb)
    plt.title("Game Frame Preview (Use pixel coordinates to verify Scoreboard ROI)")
    plt.grid(True)
    plt.show()
else:
    print("Could not extract frame.")

In [ ]:
# 6. Agent 2 (Scorekeeper): Detect audio spikes and track scoreboard changes
from src.agents.scorekeeper import ScorekeeperAgent

scorekeeper = ScorekeeperAgent(output_dir=DATA_DIR)

# Scoreboard ROI in normalized (x, y, w, h) coordinates for LBA broadcast bug
# Typical top-left overlay: x=0.03, y=0.04, w=0.28, h=0.08
scoreboard_roi = (0.03, 0.04, 0.28, 0.08)

timeline_path = scorekeeper.generate_timeline(
    video_path=DATA_DIR / "raw_video.mp4",
    audio_path=DATA_DIR / "audio.wav",
    scoreboard_roi=scoreboard_roi
)
print("Timeline events generated at:", timeline_path)

In [ ]:
# 7. Agent 3 (Archivist): Slice possession clips preceding each score change
from src.agents.archivist import ArchivistAgent

archivist = ArchivistAgent(output_dir=DATA_DIR)
clips_manifest = archivist.extract_possessions(
    video_path=DATA_DIR / "raw_video.mp4",
    timeline_events_path=timeline_path,
    lookback_sec=14.0,
    follow_through_sec=2.0
)
print(f"Extracted {len(clips_manifest)} possession clips into {archivist.clips_dir}")